# Parametric sensitivity (sIPOPT)

Given an NLP with parameters `p` that appear as right-hand sides of equality constraints (`g_i(x) = p_i`), `Problem.solve_with_sens` returns two outputs at the converged solution:

* **First-order step** `Δx ≈ ∂x*/∂p · Δp` for a user-supplied perturbation `Δp` — no resolve required.
* **Reduced Hessian** `H_R = ∇²_p L_R` (column-major, `n_params × n_params`) — the second-order curvature of the reduced problem in parameter space, used for parameter-estimation covariance / Cramér–Rao bounds.

Both are computed by one Schur-complement back-solve against the converged KKT factor — the implementation is a Rust port of upstream Ipopt's `contrib/sIPOPT/` (Pirnay, López-Negrete & Biegler 2012, [DOI 10.1007/s12532-012-0043-2](https://doi.org/10.1007/s12532-012-0043-2)).

---

## Set-up: ParametricTNLP

Minimal example from upstream's `examples/parametric_cpp/`:

$$\min_{x_1,x_2,x_3,\eta_1,\eta_2} x_1^2 + x_2^2 + x_3^2$$
subject to
$$\begin{aligned}
6 x_1 + 3 x_2 + 2 x_3 - \eta_1 &= 0 \\
\eta_2 x_1 + x_2 - x_3 - 1 &= 0 \\
\eta_1 &= p_1 \\
\eta_2 &= p_2 \\
x_1, x_2, x_3 &\ge 0.
\end{aligned}$$

The parameters `p₁, p₂` enter via the trailing two equality constraints; we'll point `solve_with_sens` at `pin_constraint_indices=[2, 3]` to mark them.

In [1]:
import numpy as np
import pounce


class ParametricNLP:
    def objective(self, x):
        return x[0] ** 2 + x[1] ** 2 + x[2] ** 2

    def gradient(self, x):
        return np.array([2 * x[0], 2 * x[1], 2 * x[2], 0.0, 0.0])

    def constraints(self, x):
        x1, x2, x3, e1, e2 = x
        return np.array([
            6 * x1 + 3 * x2 + 2 * x3 - e1,
            e2 * x1 + x2 - x3 - 1.0,
            e1,
            e2,
        ])

    def jacobianstructure(self):
        rows = np.array([0, 0, 0, 0, 1, 1, 1, 1, 2, 3], dtype=np.int64)
        cols = np.array([0, 1, 2, 3, 0, 1, 2, 4, 3, 4], dtype=np.int64)
        return rows, cols

    def jacobian(self, x):
        return np.array([
            6.0, 3.0, 2.0, -1.0,
            x[4], 1.0, -1.0, x[0],
            1.0,
            1.0,
        ])

    def hessianstructure(self):
        rows = np.array([0, 1, 2, 4], dtype=np.int64)
        cols = np.array([0, 1, 2, 0], dtype=np.int64)
        return rows, cols

    def hessian(self, x, lagrange, obj_factor):
        return np.array([
            2.0 * obj_factor,
            2.0 * obj_factor,
            2.0 * obj_factor,
            lagrange[1],
        ])


def make(p1, p2):
    prob = pounce.Problem(
        n=5, m=4, problem_obj=ParametricNLP(),
        lb=[0.0, 0.0, 0.0, -1e19, -1e19],
        ub=[1e19, 1e19, 1e19, 1e19, 1e19],
        cl=[0.0, 0.0, p1, p2],
        cu=[0.0, 0.0, p1, p2],
    )
    prob.add_option("tol", 1e-10)
    prob.add_option("print_level", 0)
    prob.add_option("sb", "yes")
    return prob


X0 = np.array([0.15, 0.15, 0.0, 0.0, 0.0])

## 1. First-order sensitivity step

Nominal parameters `p = (5, 1)`, small perturbation `Δp = (0.05, 0)`. Compare the linear prediction `x* + Δx_sens` against a fresh resolve at `p + Δp`.

We deliberately stay in the interior of the active-set polytope: at large `|Δp|` the variable `x[2]` would cross its lower bound and the linear prediction stops being valid — that's the active-set boundary handled by `sens_boundcheck=True` — see **Section 6** below.

In [2]:
x_nom, info = make(p1=5.0, p2=1.0).solve_with_sens(
    x0=X0,
    pin_constraint_indices=[2, 3],
    deltas=[0.05, 0.0],
)
print(f"converged: {info['status_msg']}, iters={info['iter_count']}")
print(f"x*           = {x_nom}")
print(f"Δx_sens      = {info['dx']}")
print(f"x* + Δx_sens = {x_nom + info['dx']}")

x_resolve, _ = make(p1=5.05, p2=1.0).solve(x0=X0)
print(f"resolve      = {x_resolve}")
print(f"|sens - resolve| = {np.max(np.abs((x_nom + info['dx']) - x_resolve)):.2e}")

converged: Solve_Succeeded, iters=9
x*           = [0.63265306 0.3877551  0.02040816 5.         1.        ]
Δx_sens      = [0.00561224 0.00102041 0.00663265 0.05       0.        ]
x* + Δx_sens = [0.63826531 0.38877551 0.02704082 5.05       1.        ]
resolve      = [0.63826531 0.38877551 0.02704082 5.05       1.        ]
|sens - resolve| = 4.53e-12


The sensitivity step matches the resolve to within ~1e-13 — first-order accuracy at a small Δp = 0.05 step, well inside the active-set polytope. `Δx` is the primal-only slice; `dx_full` carries the full KKT-space step (primals + slacks + duals) for cross-checking.

**Why use it instead of just resolving?** The sensitivity step is a single back-solve against the already-factored KKT matrix — no Newton iterations, no fresh Hessian/Jacobian evaluations. For problems where the factor cost dominates, it's effectively free.

## 2. Finite-difference convergence

Sanity check: as `Δp → 0`, the sensitivity step should approach the FD slope at first-order rate.

In [3]:
x_nom, _ = make(p1=5.0, p2=1.0).solve(x0=X0)

steps = [1e-1, 1e-2, 1e-3, 1e-4]
print(f"{'step':>10}  {'|FD - sens|':>14}")
for h in steps:
    _, info = make(p1=5.0, p2=1.0).solve_with_sens(
        x0=X0, pin_constraint_indices=[2, 3], deltas=[h, 0.0],
    )
    x_pert, _ = make(p1=5.0 + h, p2=1.0).solve(x0=X0)
    dx_fd = x_pert - x_nom
    err = np.max(np.abs(dx_fd[:3] - info['dx'][:3]))
    print(f"{h:>10.0e}  {err:>14.2e}")

      step     |FD - sens|
     1e-01        1.52e-11
     1e-02        1.07e-14
     1e-03        3.61e-14
     1e-04        4.11e-15


Error scales like `O(h²)` (quadratic in the step) — exactly the residual of the first-order Taylor expansion. The floor at small `h` is the IPM convergence tolerance (`tol=1e-10`) leaking into the FD numerator.

## 3. Reduced Hessian

`H_R = ∇²_p L_R` is the curvature of the reduced problem (objective Lagrangian projected onto the parameter manifold). For parameter estimation, `H_R⁻¹ ≈ Cov(p̂)` near the optimum (Cramér–Rao bound, given the unit-weighted least-squares interpretation).

In [4]:
_, info = make(p1=5.0, p2=1.0).solve_with_sens(
    x0=X0,
    pin_constraint_indices=[2, 3],
    compute_reduced_hessian=True,
)
Hr = info['reduced_hessian'].reshape(2, 2, order='F')
print("Reduced Hessian H_R (2x2):")
print(Hr)
print(f"\nsymmetric? max|H - Hᵀ| = {np.max(np.abs(Hr - Hr.T)):.2e}")
eig = np.linalg.eigvalsh(0.5 * (Hr + Hr.T))
print(f"eigenvalues = {eig}")

Reduced Hessian H_R (2x2):
[[-0.06122449 -0.05830896]
 [-0.05830896 -0.41566016]]

symmetric? max|H - Hᵀ| = 6.94e-18
eigenvalues = [-0.42500624 -0.05187841]


`H_R` is symmetric by construction (it factors as `B K⁻¹ Bᵀ` with `K` the symmetric KKT block). For this NLP the reduced Hessian is **indefinite** — that just reflects that the equality-constrained problem isn't pinned to a strict local minimum in parameter space alone. For an unconstrained parameter-estimation least-squares problem the corresponding `H_R` is positive definite, and its inverse is the standard error covariance.

## 4. Both outputs at once

`with_deltas` and `with_reduced_hessian` compose — pay the factor once, harvest both.

In [5]:
_, info = make(p1=5.0, p2=1.0).solve_with_sens(
    x0=X0,
    pin_constraint_indices=[2, 3],
    deltas=[0.05, 0.0],
    compute_reduced_hessian=True,
)
print("Δx (primal):       ", info['dx'])
print("H_R (column-major):", info['reduced_hessian'])

Δx (primal):        [0.00561224 0.00102041 0.00663265 0.05       0.        ]
H_R (column-major): [-0.06122449 -0.05830896 -0.05830896 -0.41566016]


## 5. Reduced-Hessian eigendecomposition

Passing `rh_eigendecomp=True` implies `compute_reduced_hessian=True` and also returns the ascending eigenvalues plus the column-major eigenvector matrix of `H_R`. The factorization is performed by a pure-Rust cyclic-Jacobi sweep (no LAPACK dependency).

Use cases:
* **Identifiability** — small `λ_min(H_R)` flags a parameter combination the data barely constrains; the corresponding eigenvector is the linear combination of parameters that is poorly determined.
* **Confidence ellipsoids** — for least-squares problems with positive-definite `H_R`, the principal axes of the covariance ellipsoid `H_R⁻¹` are the eigenvectors and their semi-axes scale as `1/√λ_i`.


In [6]:
_, info = make(p1=5.0, p2=1.0).solve_with_sens(
    x0=X0,
    pin_constraint_indices=[2, 3],
    rh_eigendecomp=True,
)
Hr = info['reduced_hessian'].reshape(2, 2, order='F')
eigvals = info['reduced_hessian_eigenvalues']
V = info['reduced_hessian_eigenvectors'].reshape(2, 2, order='F')

print(f'eigenvalues (ascending): {eigvals}')
print(f'eigenvectors (columns):\n{V}')

# Verify Hr @ V = V @ diag(λ).
residual = Hr @ V - V * eigvals[np.newaxis, :]
print(f'\nmax |Hr V - V Λ| = {np.max(np.abs(residual)):.2e}')
print(f'orthonormality:   max |VᵀV - I| = {np.max(np.abs(V.T @ V - np.eye(2))):.2e}')


eigenvalues (ascending): [-0.42500624 -0.05187841]
eigenvectors (columns):
[[ 0.1582654   0.98739661]
 [ 0.98739661 -0.1582654 ]]

max |Hr V - V Λ| = 5.55e-17
orthonormality:   max |VᵀV - I| = 2.22e-16


The eigenvalues come back sorted ascending; the eigenvectors are orthonormal to machine precision. For this indefinite `H_R` the eigenvalues straddle zero — the negative direction is the parameter combination along which the reduced Lagrangian *decreases*, a flag that the equality-constrained optimum is a saddle in parameter space (expected here since the parameters are pinned, not optimized over).


## 6. Bound projection (`sens_boundcheck`)

The raw first-order step is a linearization of `x*(p)` and ignores the bound constraints — at large `|Δp|` it can drive a primal coordinate outside `[lb, ub]`. `sens_boundcheck=True` post-processes `Δx` with a single-pass projection onto the feasible box: any coordinate whose trial value violates a bound by more than `sens_bound_eps` is clamped to the bound.

This is **simpler than upstream sIPOPT's iterative Schur refinement** — upstream re-factorizes after each violation so the non-violating coordinates absorb the clamp under the IFT constraints. The single-pass clamp here keeps the rest of `Δx` frozen, which is cheaper but less accurate when violations are deep. For most workflows that just need the perturbed primal to stay feasible, this is enough.

Demo: pull `p₁` from 5 down to 4.5 (`Δp₁ = -0.5`). The unconstrained linear prediction drives `x[2]` to roughly `-0.046`, below its lower bound of 0.


In [7]:
# Unclamped: x[2] + dx[2] < 0.
x_nom, info_raw = make(p1=5.0, p2=1.0).solve_with_sens(
    x0=X0,
    pin_constraint_indices=[2, 3],
    deltas=[-0.5, 0.0],
)
print(f'x_nom[2]                  = {x_nom[2]:+.6f}  (lower bound = 0)')
print(f'dx[2] (unclamped)         = {info_raw["dx"][2]:+.6f}')
print(f'x_nom[2] + dx[2]          = {x_nom[2] + info_raw["dx"][2]:+.6f}  ← violates lb!')

# Clamped: pass sens_boundcheck=True.
_, info_clamped = make(p1=5.0, p2=1.0).solve_with_sens(
    x0=X0,
    pin_constraint_indices=[2, 3],
    deltas=[-0.5, 0.0],
    sens_boundcheck=True,
)
print(f'\ndx[2] (clamped)           = {info_clamped["dx"][2]:+.6f}')
print(f'x_nom[2] + dx[2] clamped  = {x_nom[2] + info_clamped["dx"][2]:+.6f}  ← exactly at lb')

# Non-violating coordinates pass through untouched.
print(f'\nnon-violating dx[0,1,3,4] unchanged: '
      f'{np.allclose(info_raw["dx"][[0,1,3,4]], info_clamped["dx"][[0,1,3,4]])}')


x_nom[2]                  = +0.020408  (lower bound = 0)
dx[2] (unclamped)         = -0.066327
x_nom[2] + dx[2]          = -0.045918  ← violates lb!

dx[2] (clamped)           = -0.020408
x_nom[2] + dx[2] clamped  = -0.000000  ← exactly at lb

non-violating dx[0,1,3,4] unchanged: True


The clamped step keeps `x_nom + Δx ∈ [lb, ub]`, with the binding coordinate pinned exactly to its bound. Use `sens_bound_eps=...` (default `1e-9`) to widen the tolerance if your bounds are noisy.


---

## See also

* Notebook `03_implicit_differentiation.ipynb` — uses `pounce.jax.solve` (a `jax.custom_vjp` wrapper around the solver) to differentiate through `x*(p)` with JAX. That path is the right choice when you want **gradients in a JAX graph**; `solve_with_sens` is the right choice when you want **the raw sIPOPT outputs** (Δx + reduced Hessian) in a NumPy workflow.
* Upstream reference: Pirnay, H., López-Negrete, R., & Biegler, L.T. (2012). *Optimal sensitivity based on IPOPT.* Mathematical Programming Computation 4(4): 307–331.